In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Number of datasets: {len(csv_files)}")

for file in csv_files:
    print(file.name)

Number of datasets: 9
churn_labels.csv
content.csv
customer_feedback.csv
customers.csv
payments.csv
subscription_plans.csv
subscriptions.csv
support_tickets.csv
viewing_activity.csv


In [ ]:
datasets = {}

for file in csv_files:
    name = file.stem
    datasets[name] = pd.read_csv(file)

print("Datasets loaded successfully:\n")

for name, df in datasets.items():
    print(f"{name:25} → {df.shape[0]:>7} rows × {df.shape[1]:>2} columns")

    

Datasets loaded successfully:

churn_labels              →    8000 rows ×  4 columns
content                   →     500 rows ×  9 columns
customer_feedback         →    5046 rows ×  6 columns
customers                 →    8048 rows × 12 columns
payments                  →  110735 rows ×  8 columns
subscription_plans        →       3 rows ×  6 columns
subscriptions             →    8875 rows × 10 columns
support_tickets           →    6120 rows × 11 columns
viewing_activity          →   66422 rows ×  9 columns


In [4]:

quality_report = []

for name, df in datasets.items():
    quality_report.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isnull().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

quality_df = pd.DataFrame(quality_report)

quality_df


,dataset,rows,columns,missing_values,duplicate_rows
0,churn_labels,8000,4,12018,0
1,content,500,9,0,0
2,customer_feedback,5046,6,0,0
3,customers,8048,12,9517,48
4,payments,110735,8,214264,0
5,subscription_plans,3,6,0,0
6,subscriptions,8875,10,21768,0
7,support_tickets,6120,11,3056,0
8,viewing_activity,66422,9,1195,0


In [5]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print("\n" + "=" * 60)
        print(f"{name}")
        print("=" * 60)

        for column, count in missing.items():
            percentage = (count / len(df)) * 100
            print(f"{column:30} {count:8} ({percentage:6.2f}%)")
            


churn_labels
churn_date                         6009 ( 75.11%)
churn_reason                       6009 ( 75.11%)

customers
age                                 322 (  4.00%)
country                             409 (  5.08%)
state_or_region                    8048 (100.00%)
city                                484 (  6.01%)
preferred_language                  254 (  3.16%)

payments
subscription_id                  110735 (100.00%)
failed_payment_reason            103529 ( 93.49%)

subscriptions
subscription_end_date              8000 ( 90.14%)
cancellation_date                  6884 ( 77.57%)
cancellation_reason                6884 ( 77.57%)

support_tickets
resolution_time_hours              1528 ( 24.97%)
customer_satisfaction_score        1528 ( 24.97%)

viewing_activity
completion_percentage               531 (  0.80%)
device_type                         664 (  1.00%)


In [6]:
checks = {
    "churn_labels": ["churned", "churn_date", "churn_reason"],
    "customers": ["customer_id", "country", "state_or_region", "city"],
    "payments": ["payment_id", "customer_id", "subscription_id", "payment_status"],
    "subscriptions": ["subscription_id", "customer_id", "subscription_status"],
}

for dataset_name, columns in checks.items():
    df = datasets[dataset_name]

    print("\n" + "=" * 70)
    print(dataset_name.upper())
    print("=" * 70)

    for column in columns:
        if column in df.columns:
            print(f"\n--- {column} ---")
            print(df[column].value_counts(dropna=False).head(10))


CHURN_LABELS

--- churned ---
churned
False    6009
True     1991
Name: count, dtype: int64

--- churn_date ---
churn_date
NaN           6009
2026-06-25      19
2025-12-16      16
2026-01-22      15
2025-10-30      15
2026-04-30      14
2026-01-31      14
2026-01-09      14
2025-10-17      14
2026-02-20      14
Name: count, dtype: int64

--- churn_reason ---
churn_reason
NaN                                 6009
Too expensive                        365
Found alternative platform           284
Not enough content                   217
No longer needed                     212
Technical issues                     182
Household budget cuts                145
Content quality decline              139
Payment issues                       139
Poor customer service experience     127
Name: count, dtype: int64

CUSTOMERS

--- customer_id ---
customer_id
CUST000621    2
CUST003050    2
CUST003753    2
CUST001035    2
CUST007985    2
CUST001766    2
CUST001212    2
CUST000424    2
CUST007827    2
C

In [7]:
print("CUSTOMER ID CHECK")
print("=" * 50)

customer_ids = set(datasets["customers"]["customer_id"].dropna())

for dataset_name in [
    "churn_labels",
    "payments",
    "subscriptions",
    "support_tickets",
    "viewing_activity",
    "customer_feedback"
]:
    df = datasets[dataset_name]

    if "customer_id" in df.columns:
        ids = set(df["customer_id"].dropna())
        unmatched = ids - customer_ids

        print(
            f"{dataset_name:20} "
            f"unique IDs: {len(ids):6} | "
            f"unmatched: {len(unmatched):6}"
        )

CUSTOMER ID CHECK
churn_labels         unique IDs:   8000 | unmatched:      0
payments             unique IDs:   8000 | unmatched:      0
subscriptions        unique IDs:   8000 | unmatched:      0
support_tickets      unique IDs:   4408 | unmatched:      0
viewing_activity     unique IDs:   7980 | unmatched:      0
customer_feedback    unique IDs:   3561 | unmatched:      0
